<a href="https://colab.research.google.com/github/VeenusDadri/training/blob/master/advance_pyspark/Spark_SQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
import os

In [26]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('Spark SQL').getOrCreate()
sc = spark.sparkContext

In [27]:
print(os.path.exists("/content/Students_Grading_Dataset.csv"))


True


In [28]:
df=spark.read.csv("/content/Students_Grading_Dataset.csv",header=True,inferSchema=True)
df.printSchema()

root
 |-- Student_ID: string (nullable = true)
 |-- First_Name: string (nullable = true)
 |-- Last_Name: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Department: string (nullable = true)
 |-- Attendance (%): double (nullable = true)
 |-- Midterm_Score: double (nullable = true)
 |-- Final_Score: double (nullable = true)
 |-- Assignments_Avg: double (nullable = true)
 |-- Quizzes_Avg: double (nullable = true)
 |-- Participation_Score: double (nullable = true)
 |-- Projects_Score: double (nullable = true)
 |-- Total_Score: double (nullable = true)
 |-- Grade: string (nullable = true)
 |-- Study_Hours_per_Week: double (nullable = true)
 |-- Extracurricular_Activities: string (nullable = true)
 |-- Internet_Access_at_Home: string (nullable = true)
 |-- Parent_Education_Level: string (nullable = true)
 |-- Family_Income_Level: string (nullable = true)
 |-- Stress_Level (1-10): integer (nullable = 

In [29]:
df.count()

5000

In [30]:
df.show(10)

+----------+----------+---------+--------------------+------+---+-----------+--------------+-------------+-----------+---------------+-----------+-------------------+--------------+-----------+-----+--------------------+--------------------------+-----------------------+----------------------+-------------------+-------------------+---------------------+
|Student_ID|First_Name|Last_Name|               Email|Gender|Age| Department|Attendance (%)|Midterm_Score|Final_Score|Assignments_Avg|Quizzes_Avg|Participation_Score|Projects_Score|Total_Score|Grade|Study_Hours_per_Week|Extracurricular_Activities|Internet_Access_at_Home|Parent_Education_Level|Family_Income_Level|Stress_Level (1-10)|Sleep_Hours_per_Night|
+----------+----------+---------+--------------------+------+---+-----------+--------------+-------------+-----------+---------------+-----------+-------------------+--------------+-----------+-----+--------------------+--------------------------+-----------------------+---------------

Register the DataFrame as a Temporary Table

In [31]:
#register the DataFrame As a TEmporary Table
df.createOrReplaceTempView("my_table")

Perfrom sql-like Queries

In [32]:
result=spark.sql("Select*from my_table where Final_Score>=90")
print(result.count())
result.show()

776
+----------+----------+---------+--------------------+------+---+-----------+--------------+-------------+-----------+---------------+-----------+-------------------+--------------+-----------+-----+--------------------+--------------------------+-----------------------+----------------------+-------------------+-------------------+---------------------+
|Student_ID|First_Name|Last_Name|               Email|Gender|Age| Department|Attendance (%)|Midterm_Score|Final_Score|Assignments_Avg|Quizzes_Avg|Participation_Score|Projects_Score|Total_Score|Grade|Study_Hours_per_Week|Extracurricular_Activities|Internet_Access_at_Home|Parent_Education_Level|Family_Income_Level|Stress_Level (1-10)|Sleep_Hours_per_Night|
+----------+----------+---------+--------------------+------+---+-----------+--------------+-------------+-----------+---------------+-----------+-------------------+--------------+-----------+-----+--------------------+--------------------------+-----------------------+-----------

In [33]:
avg_Attendence_by_Age = spark.sql("Select Age , AVG(`Attendance (%)`) as avg_Attendence from my_table GROUP BY AGE")
print(avg_Attendence_by_Age.count())
avg_Attendence_by_Age.show()

7
+---+-----------------+
|Age|   avg_Attendence|
+---+-----------------+
| 22|75.83089418777934|
| 20|75.18996604414262|
| 19|74.61491419656788|
| 23|75.88986132511556|
| 24|75.47069337442221|
| 21|75.08896084337343|
| 18|75.91753623188406|
+---+-----------------+



In [34]:
df.createOrReplaceTempView("new")

In [35]:
spark.catalog.tableExists("new")

True

In [36]:
spark.catalog.dropTempView("new")

True

In [37]:
spark.catalog.tableExists("new")

False

USING ADVANCE SQL OPERATIONS FOR DATA ANALYSIS
SUBQURIES

In [38]:
Student_data=[(1,"Veenus"),(2,"Krishna"),(3,"Shiv"),(4,"Nannu"),(5,"Suman"),(6,"Ritu"),(7,"Himanshu"),(8,"Prem")]
Student=spark.createDataFrame(Student_data,["id","name"])

In [39]:
Student.show()

+---+--------+
| id|    name|
+---+--------+
|  1|  Veenus|
|  2| Krishna|
|  3|    Shiv|
|  4|   Nannu|
|  5|   Suman|
|  6|    Ritu|
|  7|Himanshu|
|  8|    Prem|
+---+--------+



In [56]:
deparment_data=[(1,"cse",2002),(2,"eee",2003),(3,"ese",2002),(4,"it",2004),(5,"mech",2004),(6,"civil",2001),(7,"bio",2004),(8,"chemical",2003)]
deparment=spark.createDataFrame(deparment_data,["id","course","YEAR"])

In [57]:
deparment.show()

+---+--------+----+
| id|  course|YEAR|
+---+--------+----+
|  1|     cse|2002|
|  2|     eee|2003|
|  3|     ese|2002|
|  4|      it|2004|
|  5|    mech|2004|
|  6|   civil|2001|
|  7|     bio|2004|
|  8|chemical|2003|
+---+--------+----+



In [58]:
Student.createOrReplaceTempView("Student")
deparment.createOrReplaceTempView("deparment")

In [60]:
#subQuery to find student with deparment above Year
result=spark.sql("SELECT name FROM student WHERE id IN (SELECT id FROM deparment WHERE YEAR>2003)")
result.show()

+--------+
|    name|
+--------+
|   Nannu|
|   Suman|
|Himanshu|
+--------+



WINDOWS FUNCTION

In [61]:
from pyspark.sql.window import Window
from pyspark.sql import functions as f

In [62]:
student_depament=spark.sql("SELECT deparment.course,student.name,student.id,deparment.YEAR FROM student JOIN  deparment ON student.id=deparment.id")
student_depament.show()

+--------+--------+---+----+
|  course|    name| id|YEAR|
+--------+--------+---+----+
|     cse|  Veenus|  1|2002|
|     eee| Krishna|  2|2003|
|     ese|    Shiv|  3|2002|
|      it|   Nannu|  4|2004|
|    mech|   Suman|  5|2004|
|   civil|    Ritu|  6|2001|
|     bio|Himanshu|  7|2004|
|chemical|    Prem|  8|2003|
+--------+--------+---+----+



In [63]:
Window_spec = Window.partitionBy("course").orderBy(f.desc("YEAR"))

In [65]:
student_depament.withColumn("rank", f.rank().over(Window_spec)).show()

+--------+--------+---+----+----+
|  course|    name| id|YEAR|rank|
+--------+--------+---+----+----+
|     bio|Himanshu|  7|2004|   1|
|chemical|    Prem|  8|2003|   1|
|   civil|    Ritu|  6|2001|   1|
|     cse|  Veenus|  1|2002|   1|
|     eee| Krishna|  2|2003|   1|
|     ese|    Shiv|  3|2002|   1|
|      it|   Nannu|  4|2004|   1|
|    mech|   Suman|  5|2004|   1|
+--------+--------+---+----+----+

